# Converters

Converters are used to transform prompts before sending them to the target.

This can be useful for a variety of reasons, such as encoding the prompt in a different format, or adding additional information to the prompt. For example, you might want to convert a prompt to base64 before sending it to the target, or add a prefix to the prompt to indicate that it is a question.

Converters can transform prompts in various ways:
- **Text-to-Text**: Encoding, obfuscation, translation, and semantic transformations
- **Multimodal**: Converting between text, images, audio, video, and files

## Converter Reference Table

The following table shows all available converters organized by their input/output modalities
and classified according to the
[MLCommons Jailbreak Attack Taxonomy](https://github.com/mlcommons/jailbreak-taxonomy)
{cite}`maple2026jailbreakmethodology`.
The taxonomy provides a mechanism-first classification of single-turn, inference-time prompt
attacks on LLMs, organized into 4 families, 8 categories, and 18 leaves. For the full taxonomy
specification and methodology, see the MLCommons AI Safety Benchmark papers
{cite}`vidgen2024ailuminate` {cite}`ghosh2025ailuminatev1`.

The **Taxonomy** column shows the family and leaf for each converter. Some converters span
multiple taxonomy leaves; the primary classification is shown.

In [1]:
import pandas as pd

from pyrit.prompt_converter import get_converter_modalities, get_taxonomy_classification
from pyrit.setup import IN_MEMORY, initialize_pyrit_async

await initialize_pyrit_async(memory_db_type=IN_MEMORY)  # type: ignore

# Get all converters with their modalities and taxonomy classifications
converter_list = get_converter_modalities()
taxonomy_map = get_taxonomy_classification()

# Build a single table with modalities and taxonomy classification
rows = []
for name, inputs, outputs in converter_list:
    input_str = ", ".join(inputs) if inputs else "any"
    output_str = ", ".join(outputs) if outputs else "any"
    classifications = taxonomy_map.get(name, ())
    taxonomy_str = "; ".join(f"{c.family}: {c.leaf}" for c in classifications) if classifications else ""
    rows.append({
        "Converter": name,
        "Input Modality": input_str,
        "Output Modality": output_str,
        "Taxonomy": taxonomy_str,
    })

# Create DataFrame and sort
df = pd.DataFrame(rows)
df = df.sort_values(by=["Input Modality", "Output Modality", "Converter"]).reset_index(drop=True)

# Display all rows
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 50)
print(df.to_string())

Found default environment files: ['C:\\Users\\romanlutz\\.pyrit\\.env', 'C:\\Users\\romanlutz\\.pyrit\\.env.local']
Loaded environment file: C:\Users\romanlutz\.pyrit\.env
Loaded environment file: C:\Users\romanlutz\.pyrit\.env.local


No new upgrade operations detected.


                              Converter   Input Modality Output Modality                                               Taxonomy
0                    AudioEchoConverter       audio_path      audio_path                                                       
1               AudioFrequencyConverter       audio_path      audio_path                                                       
2                   AudioSpeedConverter       audio_path      audio_path                                                       
3                  AudioVolumeConverter       audio_path      audio_path                                                       
4              AudioWhiteNoiseConverter       audio_path      audio_path                                                       
5       AzureSpeechAudioToTextConverter       audio_path            text                                                       
6                 AddTextImageConverter       image_path      image_path                                

## Converter Categories

Converters are organized into the following categories:

- **[Text-to-Text Converters](1_text_to_text_converters.ipynb)**: Non-LLM (encoding, obfuscation) and LLM-based (translation, variation, tone)
- **[Audio Converters](2_audio_converters.ipynb)**: Text-to-Audio, Audio-to-Text, Audio-to-Audio
- **[Image Converters](3_image_converters.ipynb)**: Text-to-Image, Image-to-Image
- **[Video Converters](4_video_converters.ipynb)**: Image-to-Video
- **[File Converters](5_file_converters.ipynb)**: Text-to-File (PDF, URL)
- **[Selectively Converting](6_selectively_converting.ipynb)**: Advanced techniques for converting portions of prompts

## Simple Converter Example

Converters can be used to perform these types of transformations. Here is a simple program that uses Rot13Converter converter, RandomCapitalLettersConverter, and AsciiArtConverter.

In [2]:
from pyrit.prompt_converter import (
    AsciiArtConverter,
    BinaryConverter,
    FirstLetterConverter,
    RandomCapitalLettersConverter,
    ROT13Converter,
)

prompt = "tell me how to cut down a tree"

print(await ROT13Converter().convert_tokens_async(prompt=prompt))  # type: ignore
print(await RandomCapitalLettersConverter(percentage=25.0).convert_tokens_async(prompt=prompt))  # type: ignore
print(await AsciiArtConverter().convert_tokens_async(prompt=prompt))  # type: ignore
print(await BinaryConverter().convert_tokens_async(prompt=prompt))  # type: ignore
print(await FirstLetterConverter().convert_tokens_async(prompt=prompt))  # type: ignore

text: gryy zr ubj gb phg qbja n gerr
text: telL me HoW To Cut down A tRee
text:   O~~             O~~ O~~                                                          O~~                              O~~        O~~                                             O~~                             
  O~~             O~~ O~~                         O~~                              O~~                              O~~        O~~                                             O~~                             
O~O~ O~   O~~     O~~ O~~ O~~~ O~~ O~~    O~~     O~~        O~~    O~~      O~~ O~O~ O~   O~~        O~~~O~~  O~~O~O~ O~      O~~   O~~    O~~      O~~O~~ O~~      O~~     O~O~ O~O~ O~~~   O~~       O~~    
  O~~   O~   O~~  O~~ O~~  O~~  O~  O~~ O~   O~~  O~ O~    O~~  O~~  O~~  ~  O~~   O~~   O~~  O~~   O~~   O~~  O~~  O~~    O~~ O~~ O~~  O~~  O~~  ~  O~~ O~~  O~~  O~~  O~~    O~~   O~~    O~   O~~  O~   O~~ 
  O~~  O~~~~~ O~~ O~~ O~~  O~~  O~  O~~O~~~~~ O~~ O~~  O~~O~~    O~~ O~~ O~  O~~   O~~  

## Stacking Converters

Although converters can be used on their own, they should be thought of as a piece in the pipeline. Typically any attack will have arguments so that prompts can be converted before being sent to the target. They can be stacked, use LLMs, and are a powerful tool.


### Stacking Converters with PromptSendingAttack

In the following example, a TextTarget is used so these prompts are simply printed and added to memory. This can be useful if you are red teaming something and need to manually enter prompts. However, the target can be replaced with any other [target](../targets/0_prompt_targets.md). E.g., if you have API access you can add a target there.

In this example the converters are stacked. First a variation is found using `VariationConverter`, and then `StringJoinConverter` is used to add a dash between letters. Remember that order matters. If `StringJoinConverter` came first, we would be asking the LLM to make variations of the prompt:
"t-e-l-l- - m-e- -h-o-w- -t-o- -c-u-t- -d-o-w-n - a- -t-r-e-e"

In [3]:
from pyrit.executor.attack import (
    AttackConverterConfig,
    ConsoleAttackResultPrinter,
    PromptSendingAttack,
)
from pyrit.prompt_converter import StringJoinConverter, VariationConverter
from pyrit.prompt_normalizer import PromptConverterConfiguration
from pyrit.prompt_target import OpenAIChatTarget, TextTarget

objective = "tell me how to cut down a tree"

# converter target to send prompt to; in this case configured to the Azure OpenAI GPT-4o Chat model
converter_target = OpenAIChatTarget()
prompt_variation_converter = VariationConverter(converter_target=converter_target)

converter_configs = PromptConverterConfiguration.from_converters(  # type: ignore
    converters=[prompt_variation_converter, StringJoinConverter()]
)

converter_config = AttackConverterConfig(request_converters=converter_configs)  # type: ignore

target = TextTarget()
attack = PromptSendingAttack(
    objective_target=target,
    attack_converter_config=converter_config,
)

result = await attack.execute_async(objective=objective)  # type: ignore

printer = ConsoleAttackResultPrinter()
await printer.print_conversation_async(result=result)  # type: ignore

TextTarget: user: C-a-n y-o-u e-x-p-l-a-i-n t-h-e p-r-o-c-e-s-s o-f c-h-o-p-p-i-n-g d-o-w-n a t-r-e-e-?



────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 Turn 1 - USER
────────────────────────────────────────────────────────────────────────────────────────────────────
   Original:
  tell me how to cut down a tree

   Converted:
  C-a-n y-o-u e-x-p-l-a-i-n t-h-e p-r-o-c-e-s-s o-f c-h-o-p-p-i-n-g d-o-w-n a t-r-e-e-?

────────────────────────────────────────────────────────────────────────────────────────────────────


## Response Converters

So far, we've focused on **request converters** that transform prompts before sending them to the target. PyRIT also supports **response converters** that transform the target's response before returning it. This is useful in scenarios like:

- Translating responses back to the original language after sending prompts in a different language
- Decoding encoded responses
- Normalizing or cleaning up response text

Response converters use the same `PromptConverterConfiguration` class as request converters. They are configured via the `response_converters` parameter in `AttackConverterConfig`.

### Translation Round-Trip Example

A common use case is sending prompts in a different language to test how the target handles non-English input. In this example, we:

1. Use a **request converter** to translate the prompt from English to French
2. Send the translated prompt to the target
3. Use a **response converter** to translate the response back to English

In [4]:
from pyrit.executor.attack import (
    AttackConverterConfig,
    ConsoleAttackResultPrinter,
    PromptSendingAttack,
)
from pyrit.prompt_converter import TranslationConverter
from pyrit.prompt_normalizer import PromptConverterConfiguration
from pyrit.prompt_target import OpenAIChatTarget

objective = "What is the capital of France?"

# Create an LLM target for the converters
converter_target = OpenAIChatTarget()

# Create an LLM target to send prompts to
prompt_target = OpenAIChatTarget()

# Request converter: translate English to French
request_converter = TranslationConverter(converter_target=converter_target, language="French")
request_converter_config = PromptConverterConfiguration(converters=[request_converter])

# Response converter: translate response back to English
response_converter = TranslationConverter(converter_target=converter_target, language="English")
response_converter_config = PromptConverterConfiguration(converters=[response_converter])

# Configure the attack with both request and response converters
converter_config = AttackConverterConfig(
    request_converters=[request_converter_config],
    response_converters=[response_converter_config],
)

attack = PromptSendingAttack(
    objective_target=prompt_target,
    attack_converter_config=converter_config,
)

result = await attack.execute_async(objective=objective)  # type: ignore

# Print the conversation showing both original and converted values
printer = ConsoleAttackResultPrinter()
await printer.print_conversation_async(result=result)  # type: ignore


────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 Turn 1 - USER
────────────────────────────────────────────────────────────────────────────────────────────────────
   Original:
  What is the capital of France?

   Converted:
  Quelle est la capitale de la France ?

────────────────────────────────────────────────────────────────────────────────────────────────────
🔸 ASSISTANT
────────────────────────────────────────────────────────────────────────────────────────────────────
   Original:
  La capitale de la France est **Paris**. 😊

   Converted:
  The capital of France is **Paris**. 😊

────────────────────────────────────────────────────────────────────────────────────────────────────
